# V4 — Fine-tune Bi-encoder (Kaggle T4)

**Model:** `BAAI/bge-m3` | **Loss:** `MultipleNegativesRankingLoss`

> **Fixes:** `sentence-transformers==2.7.0`, không dùng `checkpoint_path`, build `expected_citations` từ `van_ban/dieu/khoan`

In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────
DATASET_NAME = "vietnamese-legal-rag"   # ← đổi thành tên dataset Kaggle
# ──────────────────────────────────────────────────────────────

In [ ]:
import subprocess
result = subprocess.run(
    ["pip", "install", "-q", "sentence-transformers==2.7.0", "faiss-cpu"],
    capture_output=True, text=True
)
print(result.stderr[-300:] if result.stderr else "")
print("✓ Packages installed")

In [ ]:
import os, json
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch, numpy as np, faiss
from pathlib import Path
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses

DATA_DIR    = Path(f"/kaggle/input/{DATASET_NAME}")
WORK_DIR    = Path("/kaggle/working")
TRAIN_FILE  = DATA_DIR / "train.jsonl"
DEV_FILE    = DATA_DIR / "dev.jsonl"
SAVE_PATH   = WORK_DIR / "bi_bge_m3_ft"
FAISS_PATH  = WORK_DIR / "faiss_v4.index"

BASE_MODEL   = "BAAI/bge-m3"
EPOCHS       = 3
BATCH_SIZE   = 8
LR           = 2e-5
TOP_N        = 100
ENCODE_BATCH = 64
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

assert TRAIN_FILE.exists(), f"Không tìm thấy {TRAIN_FILE}"
assert DEV_FILE.exists(),   f"Không tìm thấy {DEV_FILE}"
SAVE_PATH.mkdir(parents=True, exist_ok=True)

print(f"GPU  : {torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'CPU'}")
if DEVICE == "cuda":
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"Batch: {BATCH_SIZE} | Epochs: {EPOCHS} | LR: {LR}")

## 1. Helpers

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def build_corpus(dev_file):
    seen, corpus = set(), []
    for row in load_jsonl(dev_file):
        key = (row.get("van_ban",""), str(row.get("dieu","")),
               str(row.get("khoan","")), str(row.get("chunk_index","")))
        if key not in seen:
            seen.add(key)
            corpus.append({"passage": row["passage"], "meta": key})
    return corpus

def get_ec(item):
    """Build expected_citations — từ field sẵn hoặc từ van_ban/dieu/khoan."""
    if "expected_citations" in item:
        return item["expected_citations"]
    return [{"van_ban": item.get("van_ban",""),
             "dieu":    item.get("dieu",""),
             "khoan":   item.get("khoan","")}]

def is_hit(idx, ec, corpus):
    if not (0 <= idx < len(corpus)): return False
    meta = corpus[idx]["meta"]
    return any(
        e.get("van_ban","") == meta[0]
        and str(e.get("dieu","")) == meta[1]
        and str(e.get("khoan","")) == meta[2]
        for e in ec
    )

def compute_metrics(per_query, top_n):
    n = len(per_query)
    return {
        "Recall@1":        round(sum(r["hit@1"] for r in per_query) / n, 4),
        "Recall@5":        round(sum(r["hit@5"] for r in per_query) / n, 4),
        f"Recall@{top_n}": round(sum(1 for r in per_query if r["rank"] > 0) / n, 4),
        "MRR@10":          round(sum(1/r["rank"] for r in per_query if 0 < r["rank"] <= 10) / n, 4),
    }
print("Helpers ✓")

## 2. Training Data

In [ ]:
raw = load_jsonl(TRAIN_FILE)
pos_pairs = [r for r in raw if r.get("label", 1) == 1]
print(f"Train pairs: {len(pos_pairs):,}")

train_examples   = [InputExample(texts=[r["query"], r["passage"]]) for r in pos_pairs]
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
steps_per_epoch  = len(train_dataloader)
print(f"DataLoader: {steps_per_epoch} steps/epoch × {EPOCHS} = {steps_per_epoch*EPOCHS} total")

## 3. Fine-tune

In [ ]:
if (SAVE_PATH / "config.json").exists():
    print(f"✅ Loading existing model from {SAVE_PATH}")
    bi_model = SentenceTransformer(str(SAVE_PATH), device=DEVICE)
else:
    print(f"Loading {BASE_MODEL}...")
    bi_model = SentenceTransformer(BASE_MODEL, device=DEVICE)

    try:
        bi_model._first_module().auto_model.gradient_checkpointing_enable()
        print("Gradient checkpointing: ✓")
    except Exception as e:
        print(f"Gradient checkpointing: skip ({e})")

    train_loss = losses.MultipleNegativesRankingLoss(bi_model)
    warmup = int(steps_per_epoch * EPOCHS * 0.1)
    print(f"Training: {EPOCHS}ep | {steps_per_epoch} steps | warmup={warmup} | AMP=True")

    bi_model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=EPOCHS,
        warmup_steps=warmup,
        optimizer_params={"lr": LR},
        use_amp=True,
        show_progress_bar=True,
        # ⚠️ Không dùng checkpoint_path — tránh _nested_gather bug
    )
    bi_model.save(str(SAVE_PATH))
    print(f"\n✅ Saved → {SAVE_PATH}")

## 4. Build FAISS

In [ ]:
corpus = build_corpus(DEV_FILE)
print(f"Corpus: {len(corpus):,} passages")

if FAISS_PATH.exists():
    index = faiss.read_index(str(FAISS_PATH))
    print(f"FAISS loaded: {index.ntotal} ✓")
else:
    print("Encoding corpus...")
    embs = bi_model.encode(
        [d["passage"] for d in corpus], batch_size=ENCODE_BATCH,
        normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True
    ).astype("float32")
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    faiss.write_index(index, str(FAISS_PATH))
    print(f"FAISS built: {index.ntotal} vectors ✓")

## 5. Evaluate — Decision Gate

In [ ]:
eval_qa = [r for r in load_jsonl(DEV_FILE) if r.get("label", 1) == 1]
print(f"Eval: {len(eval_qa):,} queries")

# Debug: in ra 1 row để kiểm tra keys
print("Keys trong dev.jsonl:", list(eval_qa[0].keys()))

q_embs = bi_model.encode(
    [item["query"] for item in eval_qa], batch_size=ENCODE_BATCH,
    normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True
).astype("float32")
_, I = index.search(q_embs, TOP_N)

per_query = []
for item, top_ids_arr in tqdm(zip(eval_qa, I), total=len(eval_qa)):
    top_ids = top_ids_arr.tolist()
    ec = get_ec(item)   # ← tự build nếu không có expected_citations
    rank = next((r for r, idx in enumerate(top_ids, 1) if is_hit(idx, ec, corpus)), -1)
    per_query.append({
        "rank":  rank,
        "hit@1": int(any(is_hit(i, ec, corpus) for i in top_ids[:1])),
        "hit@5": int(any(is_hit(i, ec, corpus) for i in top_ids[:5])),
    })

metrics = compute_metrics(per_query, TOP_N)

BAR = "─" * 52
print(f"\n{BAR}")
print("  V4 — bge-m3 Fine-tuned")
print(BAR)
for k, v in metrics.items():
    print(f"  {k:<18} {v:.4f}")

V1  = {"Recall@1": 0.514, "Recall@5": 0.781, "MRR@10": 0.619}
V23 = {"Recall@1": 0.472, "Recall@5": 0.723, "MRR@10": 0.569}
print(f"\n  {'Metric':<18} {'V1-BM25':>8} {'V2.3-Dense':>11} {'V4-FT':>8} {'Δ':>8}")
print(BAR)
for k in ["Recall@1", "Recall@5", "MRR@10"]:
    d = metrics.get(k,0) - V23.get(k,0)
    print(f"  {k:<18} {V1[k]:>8.4f} {V23[k]:>11.4f} {metrics.get(k,0):>8.4f} {d:>+7.4f} {'✅' if d>0 else '⚠️'}")

print(f"\n  Recall@{TOP_N}: {metrics.get(f'Recall@{TOP_N}',0):.4f} (ceiling)")
print(f"  {'✅ FT tốt hơn V2.3' if metrics['Recall@1']>V23['Recall@1'] else '⚠️ FT chưa vượt V2.3'}")

json.dump(metrics, open(WORK_DIR / "metrics_v4.json", "w"), indent=2)
print("\n  Saved metrics_v4.json ✓")

## 6. Zip & Download

In [ ]:
import shutil
shutil.make_archive(str(WORK_DIR / "bi_bge_m3_ft"), "zip", str(SAVE_PATH))
print("✅ bi_bge_m3_ft.zip")
print("\n── /kaggle/working/ ──")
for f in sorted(WORK_DIR.iterdir()):
    if f.is_file():
        print(f"  📄 {f.name:<35} {f.stat().st_size/1e6:>7.1f} MB")
    else:
        print(f"  📁 {f.name}/")
print("\nCopy về local:")
print("  bi_bge_m3_ft.zip → outputs/models/bi_bge_m3_ft/")
print("  faiss_v4.index   → notebooks/outputs/tmp/faiss_v4.index")